In [17]:
import os
import re
import numpy as np
import pandas as pd
import nltk
import spacy
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

nltk.download('stopwords', quiet=True)
nlp = spacy.load('en_core_web_md')
print("All imports done.")

All imports done.


In [18]:
import kagglehub
path = kagglehub.dataset_download("shantanudhakadd/email-spam-detection-dataset-classification")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\Admin\.cache\kagglehub\datasets\shantanudhakadd\email-spam-detection-dataset-classification\versions\1


In [19]:
df = pd.read_csv(
    os.path.join(path, 'spam.csv'),
    encoding='latin-1',
    usecols=[0, 1],
    names=['label', 'text'],
    skiprows=1
)

print("Shape:", df.shape)
print("Label distribution:")
print(df['label'].value_counts())
print(f"Spam ratio: {df['label'].value_counts(normalize=True)['spam']:.2%}")
df.head()

Shape: (5572, 2)
Label distribution:
label
ham     4825
spam     747
Name: count, dtype: int64
Spam ratio: 13.41%


,label,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [20]:
stop_words = set(stopwords.words('english'))

def preprocess(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    tokens = [t for t in text.split() if t not in stop_words and len(t) > 1]
    return ' '.join(tokens)

df['clean'] = df['text'].apply(preprocess)
df['label_num'] = (df['label'] == 'spam').astype(int)

print("Sample original:", df['text'].iloc[0])
print("Sample cleaned: ", df['clean'].iloc[0])
print("Avg words per message:", df['clean'].apply(lambda x: len(x.split())).mean())

Sample original: Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...
Sample cleaned:  go jurong point crazy available bugis great world la buffet cine got amore wat
Avg words per message: 8.508076094759511


In [21]:
X = df['clean']
y = df['label_num']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {len(X_train)}  |  Val: {len(X_val)}")
print(f"Train spam: {y_train.mean():.2%}  |  Val spam: {y_val.mean():.2%}")

Train: 4457  |  Val: 1115
Train spam: 13.42%  |  Val spam: 13.36%


In [22]:
bow = CountVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_bow = bow.fit_transform(X_train)
X_val_bow   = bow.transform(X_val)

tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf   = tfidf.transform(X_val)

print(f"BoW matrix shape:   {X_train_bow.shape}")
print(f"TF-IDF matrix shape: {X_train_tfidf.shape}")

BoW matrix shape:   (4457, 5000)
TF-IDF matrix shape: (4457, 5000)


In [23]:
lr_bow = LogisticRegression(max_iter=1000, random_state=42, C=1.0)
lr_bow.fit(X_train_bow, y_train)

lr_tfidf = LogisticRegression(max_iter=1000, random_state=42, C=1.0)
lr_tfidf.fit(X_train_tfidf, y_train)

print("BoW model trained:", lr_bow)
print("TF-IDF model trained:", lr_tfidf)

BoW model trained: LogisticRegression(max_iter=1000, random_state=42)
TF-IDF model trained: LogisticRegression(max_iter=1000, random_state=42)


In [24]:
def text_to_embedding(text):
    doc = nlp(text)
    vectors = [tok.vector for tok in doc if tok.has_vector]
    if not vectors:
        return np.zeros(nlp.vocab.vectors_length)
    return np.mean(vectors, axis=0)

print("Creating embeddings (this may take ~1 min)...")
X_train_emb = np.array([text_to_embedding(t) for t in X_train])
X_val_emb   = np.array([text_to_embedding(t) for t in X_val])
print(f"Embedding matrix shape: {X_train_emb.shape}")

Creating embeddings (this may take ~1 min)...
Embedding matrix shape: (4457, 300)


In [25]:
lr_emb = LogisticRegression(max_iter=1000, random_state=42, C=1.0)
lr_emb.fit(X_train_emb, y_train)
print("Embedding model trained:", lr_emb)

Embedding model trained: LogisticRegression(max_iter=1000, random_state=42)


In [26]:
models = [
    ('BoW + LR',       lr_bow,   X_val_bow),
    ('TF-IDF + LR',    lr_tfidf, X_val_tfidf),
    ('spaCy emb + LR', lr_emb,   X_val_emb),
]

rows = []
for name, model, X_v in models:
    preds = model.predict(X_v)
    proba = model.predict_proba(X_v)[:, 1]
    acc = accuracy_score(y_val, preds)
    auc = roc_auc_score(y_val, proba)
    rows.append({'Model': name, 'Accuracy': round(acc, 4), 'AUC': round(auc, 4)})
    print(f"=== {name} ===")
    print(classification_report(y_val, preds, target_names=['ham', 'spam']))

print("=== Summary ===")
pd.DataFrame(rows).set_index('Model')

=== BoW + LR ===
              precision    recall  f1-score   support

         ham       0.98      1.00      0.99       966
        spam       0.99      0.86      0.92       149

    accuracy                           0.98      1115
   macro avg       0.99      0.93      0.95      1115
weighted avg       0.98      0.98      0.98      1115

=== TF-IDF + LR ===
              precision    recall  f1-score   support

         ham       0.97      1.00      0.98       966
        spam       0.98      0.81      0.89       149

    accuracy                           0.97      1115
   macro avg       0.97      0.90      0.94      1115
weighted avg       0.97      0.97      0.97      1115

=== spaCy emb + LR ===
              precision    recall  f1-score   support

         ham       0.96      0.97      0.96       966
        spam       0.77      0.71      0.74       149

    accuracy                           0.93      1115
   macro avg       0.86      0.84      0.85      1115
weighted avg  

,Accuracy,AUC
Model,,
BoW + LR,0.9803,0.9892
TF-IDF + LR,0.9722,0.9889
spaCy emb + LR,0.9327,0.9569


In [27]:
feature_names = np.array(tfidf.get_feature_names_out())
coef = lr_tfidf.coef_[0]
top_spam = feature_names[np.argsort(coef)[-15:]][::-1]
top_ham  = feature_names[np.argsort(coef)[:15]]

print("Top 15 spam features (TF-IDF + LR):")
print(list(top_spam))
print("Top 15 ham features (TF-IDF + LR):")
print(list(top_ham))

Top 15 spam features (TF-IDF + LR):
['txt', 'call', 'claim', 'mobile', 'free', 'reply', 'www', 'uk', 'stop', 'text', 'service', 'prize', 'com', 'win', 'cash']
Top 15 ham features (TF-IDF + LR):
['ok', 'home', 'sorry', 'later', 'lt', 'gt', 'got', 'come', 'da', 'lt gt', 'lor', 'hey', 'going', 'well', 'way']


Висновок: BoW та TF-IDF показують кращі результати для короткого доменно-специфічного тексту (SMS/email), оскільки конкретні слова-маркери ("free", "win", "prize") є сильними сигналами. Ембединги краще узагальнюють семантику, але втрачають специфічний словниковий сигнал.